In [4]:
import pandas as pd
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

def generate_logistics_data(n=1000, is_historical=True):
    data = []
    carriers = ["Swift", "Delhivery", "BlueDart"]
    weather_options = ["Clear", "Heavy Rain", "Cyclone", "Fog"]
    
    for _ in range(n):
        # 1. Base Features
        carrier = random.choice(carriers)
        weather = random.choices(weather_options, weights=[60, 20, 5, 15])[0]
        shipment_value = random.randint(500, 15000)
        warehouse_load = random.randint(30, 100)  # Percentage
        traffic_density = random.uniform(0.1, 1.0) # 1.0 is gridlock
        
        # 2. Timing
        if is_historical:
            # Historical data needs past timestamps
            ref_time = fake.date_time_between(start_date="-30d", end_date="now")
        else:
            # Live data needs current/future timestamps
            ref_time = datetime.now()
            
        planned_arrival = ref_time + timedelta(hours=random.randint(4, 48))
        
        # 3. Hidden Delay Logic (The "Likelihood" Patterns)
        # We calculate a probability score that the ML model will try to find
        risk_factor = 0.05
        if weather == "Cyclone": risk_factor += 0.8
        if weather == "Heavy Rain" and carrier == "BlueDart": risk_factor += 0.4
        if warehouse_load > 85: risk_factor += 0.3
        if traffic_density > 0.8: risk_factor += 0.2
        
        # Determine actual outcome based on risk
        is_delayed = 1 if random.random() < risk_factor else 0
        
        # Assign delay minutes based on the outcome
        if is_delayed:
            if weather == "Cyclone":
                delay_min = random.randint(300, 720)
                event_type = "Natural Disaster"
            else:
                delay_min = random.randint(45, 180)
                event_type = "Operational Bottleneck"
        else:
            delay_min = random.randint(0, 15)
            event_type = "Routine"

        data.append({
            "shipment_id": fake.bothify(text='??-########'),
            "carrier": carrier,
            "weather": weather,
            "shipment_value": shipment_value,
            "warehouse_load_pct": warehouse_load,
            "traffic_density": round(traffic_density, 2),
            "planned_arrival": planned_arrival,
            "current_delay_minutes": delay_min, 
            "is_delayed": is_delayed, # <--- TARGET VARIABLE FOR ML
            "event_type": event_type,
            "last_checkpoint": fake.city() + " Hub",
            "distance_to_go_km": random.randint(10, 500),
            "shipment_priority": random.choice(["High", "Medium", "Low"])
        })
    
    return pd.DataFrame(data)

# --- EXECUTION ---

# 1. Generate Historical Data (Keep everything for training)
historical_df = generate_logistics_data(1000, is_historical=True)
historical_df.to_csv("historical_logistics.csv", index=False)

# 2. Generate Live Data (The "Test" cases)
live_stream_df = generate_logistics_data(100, is_historical=False)

# 3. DROP the "answers" before giving it to the Agent
# We save the full version to a CSV for your reference/scoring later
live_stream_df.to_csv("live_shipments_FULL.csv", index=False)

# This is what you actually feed into your Agentic Loop:
agent_input_df = live_stream_df.drop(columns=['is_delayed', 'current_delay_minutes', 'event_type'])
agent_input_df.to_csv("live_shipments_FOR_AGENT.csv", index=False)